# 03 — LightGBM Model

Builds and evaluates a LightGBM model on California M5 sales data.
Results are compared against baselines from `02_baseline.ipynb`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import lightgbm as lgb
import calendar
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import os

CACHE_FILE = "df_CA_cache.parquet"
if os.path.exists(CACHE_FILE):
    df_CA = pd.read_parquet(CACHE_FILE)
    _from_cache = True
    print(f"Loaded from cache: {df_CA.shape}")
    print(f"Memory: {df_CA.memory_usage(deep=True).sum() / 1e6:.1f} MB")
else:
    _from_cache = False
    print("No cache — run sections 1–3 below.")

## Data Loading (skip if cache loaded)

In [ ]:
if not _from_cache:
    df_sales = pd.read_csv("sales_train_validation.csv")
    df_sales_CA = df_sales[df_sales["state_id"] == "CA"].copy()
    del df_sales
    id_cols = ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]
    sales_cols = [c for c in df_sales_CA.columns if c.startswith("d_")]
    df_CA_long = df_sales_CA.melt(id_vars=id_cols, value_vars=sales_cols, var_name="d", value_name="sales")
    del df_sales_CA
    df_CA_long["d_num"] = df_CA_long["d"].str.extract(r"d_(\d+)").astype("uint16")
    df_CA_long = df_CA_long.astype({"item_id":"category","dept_id":"category","cat_id":"category",
                                     "store_id":"category","state_id":"category","d":"category","sales":"uint16"})
    df_calendar = pd.read_csv("calendar.csv")
    for col in ["event_name_1","event_type_1","event_name_2","event_type_2"]:
        df_calendar[col] = df_calendar[col].fillna("No_Event")
    df_calendar = df_calendar.astype({"wm_yr_wk":"uint16","wday":"uint8","month":"uint8","year":"uint16",
        "weekday":"category","event_name_1":"category","event_type_1":"category",
        "event_name_2":"category","event_type_2":"category","snap_CA":"bool","snap_TX":"bool","snap_WI":"bool"})
    df_calendar["date"] = pd.to_datetime(df_calendar["date"]).astype("datetime64[ms]")
    df_CA_merged = df_CA_long.merge(df_calendar, on="d", how="left")
    del df_calendar
    df_sell_prices = pd.read_csv("sell_prices.csv")
    df_sp_CA = df_sell_prices[df_sell_prices["store_id"].str.startswith("CA")].copy()
    del df_sell_prices
    df_sp_CA = df_sp_CA.astype({"store_id":"category","item_id":"category","wm_yr_wk":"uint16","sell_price":"float32"})
    df_CA = df_CA_merged.merge(df_sp_CA, on=["store_id","item_id","wm_yr_wk"], how="left")
    del df_CA_merged, df_sp_CA
    df_CA.to_parquet(CACHE_FILE, index=False)
    print(f"Cache saved. Shape: {df_CA.shape}")

## Step 1 — Feature Engineering

### 1a. Sort + Log-Transform Target

In [ ]:
df_CA = df_CA.sort_values(["item_id", "store_id", "date"]).reset_index(drop=True)
df_CA["sales_log"] = np.log1p(df_CA["sales"].astype("float32")).astype("float32")
print(f"Shape: {df_CA.shape} | Memory: {df_CA.memory_usage(deep=True).sum()/1e6:.1f} MB")

### 1b. Lag Features (log scale)

In [ ]:
grp = df_CA.groupby(["item_id","store_id"], observed=True)["sales_log"]
df_CA["lag_7_log"]  = grp.shift(7).astype("float32")
df_CA["lag_28_log"] = grp.shift(28).astype("float32")

prev_year     = df_CA["date"].dt.year - 1
lookback_days = prev_year.map(lambda y: 366 if calendar.isleap(y) else 365)
df_CA["_lag_date"] = df_CA["date"] - pd.to_timedelta(lookback_days, unit="D")
lookup = df_CA[["item_id","store_id","date","sales_log"]].rename(columns={"date":"_lag_date","sales_log":"lag_364_log"})
df_CA = df_CA.merge(lookup, on=["item_id","store_id","_lag_date"], how="left")
df_CA["lag_364_log"] = df_CA["lag_364_log"].astype("float32")
df_CA = df_CA.drop(columns=["_lag_date"])
print("Lag nulls:", df_CA[["lag_7_log","lag_28_log","lag_364_log"]].isnull().sum().to_dict())

### 1c. Rolling Features

In [ ]:
grp_log = df_CA.groupby(["item_id","store_id"], observed=True)["sales_log"]
df_CA["rolling_mean_7"]  = grp_log.transform(lambda x: x.shift(1).rolling(7,  min_periods=1).mean()).astype("float32")
df_CA["rolling_mean_28"] = grp_log.transform(lambda x: x.shift(1).rolling(28, min_periods=1).mean()).astype("float32")
df_CA["rolling_std_7"]   = grp_log.transform(lambda x: x.shift(1).rolling(7,  min_periods=1).std().fillna(0)).astype("float32")
grp_raw = df_CA.groupby(["item_id","store_id"], observed=True)["sales"]
df_CA["rolling_zero_count_7"] = grp_raw.transform(
    lambda x: (x.shift(1)==0).rolling(7, min_periods=1).sum()
).astype("float32")
print(f"Memory: {df_CA.memory_usage(deep=True).sum()/1e6:.1f} MB")

### 1d. Calendar Features

In [ ]:
df_CA["day_of_week"]  = df_CA["date"].dt.dayofweek.astype("int8")
df_CA["month"]        = df_CA["date"].dt.month.astype("int8")
df_CA["week_of_year"] = df_CA["date"].dt.isocalendar().week.astype("int8")
df_CA["is_weekend"]   = (df_CA["date"].dt.dayofweek >= 5).astype("uint8")
df_CA["day_of_month"] = df_CA["date"].dt.day.astype("int8")
print("Calendar features added.")

### 1e. Event Interaction Features

In [ ]:
df_CA["event_indicator"] = (df_CA["event_type_1"] != "No_Event").astype("uint8")
df_CA["cat_event"] = (df_CA["cat_id"].astype(str) + "_" + df_CA["event_type_1"].astype(str)).astype("category")
print("Unique cat_event values:", df_CA["cat_event"].nunique())

### 1f. Price Features

In [ ]:
grp_price = df_CA.groupby(["item_id","store_id"], observed=True)["sell_price"]
price_lag7 = grp_price.shift(7).astype("float32")
df_CA["price_change_7"] = ((df_CA["sell_price"] - price_lag7) / (price_lag7.fillna(1) + 1e-8)).astype("float32")
price_roll28 = grp_price.transform(lambda x: x.shift(1).rolling(28, min_periods=1).mean()).astype("float32")
df_CA["price_ratio_28"] = (df_CA["sell_price"] / (price_roll28.fillna(1) + 1e-8)).astype("float32")
print(f"Memory: {df_CA.memory_usage(deep=True).sum()/1e6:.1f} MB")

### 1g. Drop Null Rows + Define Features

In [ ]:
df_model = df_CA.dropna(subset=["lag_7_log","lag_28_log","lag_364_log"]).reset_index(drop=True)
print(f"df_model: {df_model.shape} | {df_model['date'].min().date()} → {df_model['date'].max().date()}")

TARGET = "sales_log"

FEATURE_COLS = [
    "lag_7_log", "lag_28_log", "lag_364_log",
    "rolling_mean_7", "rolling_mean_28", "rolling_std_7", "rolling_zero_count_7",
    "day_of_week", "month", "week_of_year", "is_weekend", "day_of_month",
    "event_indicator", "cat_event",
    "snap_CA",
    "sell_price", "price_change_7", "price_ratio_28",
    "store_id", "dept_id", "cat_id",
]
CAT_COLS = ["store_id", "dept_id", "cat_id", "cat_event"]

for col in CAT_COLS:
    df_model[col] = df_model[col].astype("category")

print(f"Features: {len(FEATURE_COLS)} | Target: {TARGET}")

## Step 2 — Train / Val Split

In [ ]:
VAL_DAYS  = 28
max_date  = df_model["date"].max()
val_start = max_date - pd.Timedelta(days=VAL_DAYS - 1)

df_train = df_model[df_model["date"] <  val_start].reset_index(drop=True)
df_val   = df_model[df_model["date"] >= val_start].reset_index(drop=True)

X_train, y_train = df_train[FEATURE_COLS], df_train[TARGET]
X_val,   y_val   = df_val[FEATURE_COLS],   df_val[TARGET]

print(f"Train: {df_train.shape[0]:,} rows | {df_train['date'].min().date()} → {df_train['date'].max().date()}")
print(f"Val:   {df_val.shape[0]:,} rows | {df_val['date'].min().date()} → {df_val['date'].max().date()}")

## Step 3 — Train LightGBM

In [ ]:
params = {
    "objective":         "regression",
    "metric":            "rmse",
    "verbosity":         -1,
    "boosting_type":     "gbdt",
    "num_leaves":        127,
    "learning_rate":     0.05,
    "feature_fraction":  0.8,
    "bagging_fraction":  0.8,
    "bagging_freq":      5,
    "min_child_samples": 50,
    "lambda_l1":         0.1,
    "lambda_l2":         0.1,
    "seed":              42,
}

dtrain = lgb.Dataset(X_train, label=y_train, categorical_feature=CAT_COLS, free_raw_data=False)
dval   = lgb.Dataset(X_val,   label=y_val,   categorical_feature=CAT_COLS, reference=dtrain, free_raw_data=False)

model = lgb.train(
    params, dtrain,
    num_boost_round=1000,
    valid_sets=[dtrain, dval],
    valid_names=["train", "val"],
    callbacks=[lgb.early_stopping(50, verbose=True), lgb.log_evaluation(100)],
)

model.save_model("lgb_model.txt")
print(f"\nBest iteration: {model.best_iteration}")
print(f"Val RMSE (log scale): {model.best_score['val']['rmse']:.4f}")

## Step 4 — Evaluate vs Baselines

In [ ]:
y_pred_log = model.predict(X_val, num_iteration=model.best_iteration)
y_pred     = np.expm1(y_pred_log).clip(0)
y_true     = np.expm1(y_val.values)

print(f"Val RMSE (original scale): {np.sqrt(np.mean((y_true - y_pred)**2)):.4f}")
print(f"Val MAE  (original scale): {np.mean(np.abs(y_true - y_pred)):.4f}")

In [ ]:
# RMSSE per series
train_sorted = df_train.sort_values(["item_id","store_id","date"])
scale = (
    train_sorted.groupby(["item_id","store_id"], observed=True)["sales"]
    .apply(lambda x: np.mean(np.diff(x.values.astype(float))**2))
    .reset_index().rename(columns={"sales":"scale"})
)
scale["scale"] = scale["scale"].clip(lower=1e-8)

val_df = df_val[["item_id","store_id","sales"]].copy()
val_df["pred"] = y_pred

mse = (
    val_df.groupby(["item_id","store_id"], observed=True)
    .apply(lambda x: np.mean((x["sales"].astype(float) - x["pred"])**2), include_groups=False)
    .reset_index().rename(columns={0:"mse"})
)
rmsse_df = mse.merge(scale, on=["item_id","store_id"])
rmsse_df["rmsse"] = np.sqrt(rmsse_df["mse"] / rmsse_df["scale"])

lgbm_mean   = rmsse_df["rmsse"].mean()
lgbm_median = rmsse_df["rmsse"].median()
print(f"LightGBM  — Mean RMSSE: {lgbm_mean:.4f} | Median RMSSE: {lgbm_median:.4f}")

In [ ]:
# Load baseline benchmarks from 02_baseline.ipynb
if os.path.exists("baseline_benchmark.csv"):
    bench = pd.read_csv("baseline_benchmark.csv", index_col=0)
    print("\nModel Comparison:")
    print("=" * 55)
    print(f"{'Model':<28} {'Mean RMSSE':>12} {'Median RMSSE':>12}")
    print("-" * 55)
    for model_name, row in bench.iterrows():
        print(f"{model_name:<28} {row['Mean RMSSE']:>12.4f} {row['Median RMSSE']:>12.4f}")
    print(f"{'LightGBM':<28} {lgbm_mean:>12.4f} {lgbm_median:>12.4f}")
    print("=" * 55)
    naive_median = bench.loc["Seasonal Naive (lag_7)", "Median RMSSE"]
    improvement  = (naive_median - lgbm_median) / naive_median * 100
    print(f"\nLightGBM improvement over Seasonal Naive: {improvement:.1f}%")
else:
    print(f"LightGBM Mean RMSSE: {lgbm_mean:.4f} | Median: {lgbm_median:.4f}")
    print("Run 02_baseline.ipynb first to see comparison.")

### Feature Importance — Top 20

In [ ]:
importance = pd.DataFrame({
    "feature":    model.feature_name(),
    "importance": model.feature_importance(importance_type="gain"),
}).sort_values("importance", ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(importance["feature"][::-1], importance["importance"][::-1], color="steelblue")
ax.set_title("Top 20 Features by Gain")
ax.set_xlabel("Total Gain")
plt.tight_layout()
plt.savefig("ca_feature_importance.png", dpi=120, bbox_inches="tight")
plt.show()
print(importance.to_string(index=False))

### Residual Analysis

In [ ]:
residuals = y_true - y_pred

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0,0].scatter(y_pred, residuals, alpha=0.05, s=1, color="steelblue")
axes[0,0].axhline(0, color="tomato", linewidth=1)
axes[0,0].set_title("Residuals vs Predicted")
axes[0,0].set_xlabel("Predicted Sales")
axes[0,0].set_ylabel("Residual")
axes[0,0].set_xlim(left=0)

axes[0,1].hist(residuals.clip(-10, 20), bins=80, color="steelblue", edgecolor="none", density=True)
axes[0,1].axvline(0,                color="tomato", linestyle="--", linewidth=1, label="Zero")
axes[0,1].axvline(residuals.mean(), color="orange", linestyle="--", linewidth=1, label=f"Mean {residuals.mean():.2f}")
axes[0,1].set_title("Residual Distribution")
axes[0,1].legend()

res_df = df_val[["date","cat_id"]].copy()
res_df["residual"] = residuals
daily_res = res_df.groupby("date")["residual"].mean()
axes[1,0].plot(daily_res.index, daily_res.values, color="steelblue", linewidth=0.8)
axes[1,0].axhline(0, color="tomato", linewidth=1)
axes[1,0].set_title("Mean Daily Residual")
axes[1,0].set_xlabel("Date")

cat_res = [res_df[res_df["cat_id"]==cat]["residual"].clip(-10,20).values
           for cat in res_df["cat_id"].cat.categories]
axes[1,1].boxplot(cat_res, labels=res_df["cat_id"].cat.categories,
                  patch_artist=True, boxprops=dict(facecolor="steelblue", alpha=0.6))
axes[1,1].axhline(0, color="tomato", linewidth=1)
axes[1,1].set_title("Residuals by Category")

plt.tight_layout()
plt.savefig("ca_residuals.png", dpi=120, bbox_inches="tight")
plt.show()
print(f"Mean residual: {residuals.mean():.4f} | Std: {residuals.std():.4f}")
print(f"Under-predict: {(residuals>0).mean():.1%} | Over-predict: {(residuals<0).mean():.1%}")

### RMSSE Distribution

In [ ]:
cat_lookup = df_val[["item_id","store_id","cat_id"]].drop_duplicates()
rmsse_cat  = (
    rmsse_df.merge(cat_lookup, on=["item_id","store_id"])
    .groupby("cat_id", observed=True)["rmsse"].median().sort_values()
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(rmsse_df["rmsse"].clip(upper=5), bins=60, color="steelblue", edgecolor="none")
axes[0].axvline(lgbm_mean,   color="tomato",  linestyle="--", label=f"Mean {lgbm_mean:.2f}")
axes[0].axvline(lgbm_median, color="orange",  linestyle="--", label=f"Median {lgbm_median:.2f}")
axes[0].set_title("RMSSE Distribution (clipped at 5)")
axes[0].set_xlabel("RMSSE")
axes[0].legend()

axes[1].bar(rmsse_cat.index.astype(str), rmsse_cat.values, color="steelblue")
axes[1].set_title("Median RMSSE by Category")
axes[1].set_ylabel("Median RMSSE")

plt.tight_layout()
plt.savefig("ca_rmsse_distribution.png", dpi=120, bbox_inches="tight")
plt.show()